# 🎯 Logistic Regression — Memprediksi Ya atau Tidak

**Modul 1: Machine Learning Fundamentals** | Notebook 2 dari 9

---

## 🎯 Apa yang Akan Kita Pelajari?

1. Apa itu **Logistic Regression** dan bedanya dengan Linear Regression
2. Bagaimana **mengolah data kategori** menjadi angka yang bisa dipahami komputer
3. Cara **mengevaluasi model klasifikasi** (Confusion Matrix, Precision, Recall)
4. Teknik menangani **data tidak seimbang** *(materi bonus)*

### ⏱️ Estimasi Waktu: ~90 menit + diskusi

---

## 💡 Analogi Sederhana

Di notebook sebelumnya, kita belajar **Linear Regression** untuk menebak **angka** (berapa banyak penjualan?).

Sekarang pertanyaannya berbeda: **Ya atau Tidak?**

Contoh:
- Apakah email ini **spam** atau bukan?
- Apakah nasabah ini akan **buka deposito** atau tidak?
- Apakah siswa ini akan **lulus** atau tidak?

**Logistic Regression** = model yang memprediksi **peluang (probabilitas)** sesuatu terjadi, lalu memutuskan Ya atau Tidak.

### Bedanya dengan Linear Regression?

| | Linear Regression | Logistic Regression |
|---|---|---|
| **Prediksi** | Angka (berapa?) | Kategori (ya/tidak) |
| **Output** | Nilai kontinu (-∞ sampai +∞) | Probabilitas (0% sampai 100%) |
| **Contoh** | Prediksi harga rumah | Prediksi lulus/tidak lulus |

**Analogi:** Linear Regression seperti menebak **nilai ujian** (0-100). Logistic Regression seperti menebak **lulus atau tidak** (Ya/Tidak) berdasarkan nilai itu.

---

## 1️⃣ Persiapan — Import Library

In [ ]:
# Import library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Library Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)

# Pengaturan tampilan
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ Semua library berhasil di-import!')

---

## 2️⃣ Memuat Data

### 📋 Tentang Dataset

Kita menggunakan dataset **Bank Marketing** dari sebuah bank di Portugal. Bank ini menelepon nasabah untuk menawarkan **deposito berjangka** (tabungan khusus dengan bunga lebih tinggi).

**Pertanyaan kita:** Berdasarkan profil nasabah, bisakah kita prediksi apakah mereka akan **setuju buka deposito** atau **menolak**?

| Kolom Penting | Keterangan |
|---------------|------------|
| `age` | Umur nasabah |
| `job` | Pekerjaan (admin, teknisi, pelajar, dll) |
| `marital` | Status pernikahan |
| `education` | Tingkat pendidikan |
| `housing` | Punya kredit rumah? (yes/no) |
| `loan` | Punya pinjaman? (yes/no) |
| `duration` | Durasi telepon terakhir (detik) |
| `y` | **TARGET** — Buka deposito? (1 = Ya, 0 = Tidak) |

In [ ]:
# Upload file CSV ke Google Colab
from google.colab import files
uploaded = files.upload()

In [ ]:
# Baca dataset
data = pd.read_csv('banking.csv')

print(f'Dataset memiliki {data.shape[0]:,} baris dan {data.shape[1]} kolom')
data.head()

---

## 3️⃣ Eksplorasi Data (EDA)

### Berapa banyak yang setuju vs menolak?

In [ ]:
# Lihat distribusi target (y)
jumlah = data['y'].value_counts()
persen = data['y'].value_counts(normalize=True) * 100

print('Distribusi Target:')
print(f'  Menolak (0): {jumlah[0]:,} nasabah ({persen[0]:.1f}%)')
print(f'  Setuju  (1): {jumlah[1]:,} nasabah ({persen[1]:.1f}%)')

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#EF5350', '#66BB6A']
jumlah.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Jumlah Nasabah', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Buka Deposito?')
axes[0].set_ylabel('Jumlah')
axes[0].set_xticklabels(['Tidak (0)', 'Ya (1)'], rotation=0)

axes[1].pie(jumlah, labels=['Tidak', 'Ya'], colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proporsi', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

⚠️ **Perhatikan!** Data kita **tidak seimbang** — yang menolak jauh lebih banyak (~89%) daripada yang setuju (~11%). Ini seperti kelas yang 9 dari 10 siswanya tidak lulus — model bisa "curang" dengan selalu menebak "tidak lulus" dan sudah benar 90%!

Kita akan tangani ini nanti. Sekarang mari eksplorasi dulu.

### Siapa yang cenderung buka deposito?

In [ ]:
# Persentase yang buka deposito berdasarkan pekerjaan
job_rate = data.groupby('job')['y'].mean().sort_values(ascending=True) * 100

plt.figure(figsize=(10, 6))
bars = plt.barh(job_rate.index, job_rate.values, color='#42A5F5')

# Warnai bar tertinggi
for i, bar in enumerate(bars):
    if job_rate.values[i] > 20:
        bar.set_color('#66BB6A')

plt.xlabel('Persentase Buka Deposito (%)', fontsize=11)
plt.ylabel('Pekerjaan', fontsize=11)
plt.title('Siapa yang Paling Sering Buka Deposito?', fontsize=14, fontweight='bold')

# Tambahkan label persentase
for i, v in enumerate(job_rate.values):
    plt.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.show()

**Temuan menarik:**
- **Pelajar (student)** dan **pensiunan (retired)** paling banyak buka deposito
- **Pekerja kerah biru (blue-collar)** paling sedikit
- Ini masuk akal — pelajar mungkin menabung uang saku, pensiunan punya dana pensiun

In [ ]:
# Distribusi umur berdasarkan keputusan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram umur
for label, color, nama in [(0, '#EF5350', 'Menolak'), (1, '#66BB6A', 'Setuju')]:
    axes[0].hist(data[data['y'] == label]['age'], bins=30, alpha=0.6,
                 color=color, label=nama, edgecolor='white')
axes[0].set_xlabel('Umur', fontsize=11)
axes[0].set_ylabel('Jumlah', fontsize=11)
axes[0].set_title('Distribusi Umur vs Keputusan', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)

# Persentase berdasarkan pendidikan
edu_rate = data.groupby('education')['y'].mean().sort_values() * 100
edu_rate.plot(kind='barh', ax=axes[1], color='#AB47BC')
axes[1].set_xlabel('Persentase Buka Deposito (%)', fontsize=11)
axes[1].set_title('Pendidikan vs Keputusan', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

---

## 4️⃣ Persiapan Data untuk Model

### Masalah: Komputer Tidak Mengerti Kata-kata!

Kolom seperti `job`, `marital`, `education` berisi **teks** ("admin", "married", dll). Tapi model Machine Learning hanya bisa memproses **angka**.

**Solusi:** Kita ubah teks menjadi angka menggunakan teknik **One-Hot Encoding**.

Contoh:
```
Marital: married  →  marital_married=1, marital_single=0, marital_divorced=0
Marital: single   →  marital_married=0, marital_single=1, marital_divorced=0
```

Seperti membuat **checklist** — untuk setiap kategori, kita centang (1) atau tidak (0).

In [ ]:
# Lihat kolom mana yang bertipe teks (object)
kolom_teks = data.select_dtypes(include='object').columns.tolist()
kolom_angka = data.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'Kolom teks ({len(kolom_teks)}): {kolom_teks}')
print(f'Kolom angka ({len(kolom_angka)}): {kolom_angka}')

In [ ]:
# Ubah kolom teks menjadi angka dengan One-Hot Encoding
# drop_first=True untuk menghindari "multicollinearity" (redundansi)
data_encoded = pd.get_dummies(data, drop_first=True)

print(f'Sebelum encoding: {data.shape[1]} kolom')
print(f'Sesudah encoding: {data_encoded.shape[1]} kolom')
print(f'\nKolom baru (10 pertama):')
print(data_encoded.columns.tolist()[:10])

### Pisahkan Fitur dan Target, lalu Bagi Data

In [ ]:
# Pisahkan fitur (X) dan target (y)
X = data_encoded.drop(columns='y')
y = data_encoded['y']

# Bagi menjadi data latihan (70%) dan data ujian (30%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f'Data latihan: {X_train.shape[0]:,} baris')
print(f'Data ujian:   {X_test.shape[0]:,} baris')

### Scaling (Menyamakan Skala Data)

Bayangkan kamu membandingkan tinggi badan (cm) dengan berat badan (kg). Angkanya sangat berbeda skalanya! **StandardScaler** mengubah semua fitur ke skala yang sama agar model tidak "bingung".

Ini juga mencegah warning *"ConvergenceWarning"* yang artinya model kesulitan menemukan solusi terbaik.

In [ ]:
# Scaling — samakan skala semua fitur
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform di data train
X_test_scaled = scaler.transform(X_test)         # HANYA transform di data test

print('✅ Data berhasil di-scaling')
print(f'Contoh sebelum scaling: {X_train.iloc[0, :3].values}')
print(f'Contoh sesudah scaling: {X_train_scaled[0, :3].round(3)}')

---

## 5️⃣ Melatih Model Logistic Regression

### Bagaimana Logistic Regression Bekerja?

1. Model menghitung **skor** berdasarkan semua fitur (seperti Linear Regression)
2. Skor ini diubah menjadi **probabilitas** (0% - 100%) menggunakan fungsi **sigmoid**
3. Jika probabilitas > 50% → prediksi **Ya (1)**, jika tidak → **Tidak (0)**

**Analogi:** Seperti guru yang menilai apakah siswa lulus:
- Lihat nilai ujian, kehadiran, tugas (= fitur)
- Hitung persentase kemungkinan lulus (= probabilitas)
- Jika di atas 50% → Lulus! ✅

In [ ]:
# Buat dan latih model Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print('✅ Model berhasil dilatih!')

In [ ]:
# Prediksi pada data test
y_pred = model.predict(X_test_scaled)

# Lihat juga probabilitasnya
y_pred_proba = model.predict_proba(X_test_scaled)

# Tampilkan beberapa contoh
contoh = pd.DataFrame({
    'Asli': y_test.values[:10],
    'Prediksi': y_pred[:10],
    'Prob. Tidak (%)': (y_pred_proba[:10, 0] * 100).round(1),
    'Prob. Ya (%)': (y_pred_proba[:10, 1] * 100).round(1),
    'Benar?': ['✅' if a == p else '❌' for a, p in zip(y_test.values[:10], y_pred[:10])]
})

print('Contoh Prediksi (10 data pertama):')
contoh

---

## 6️⃣ Evaluasi Model

### Akurasi — Berapa Persen yang Benar?

In [ ]:
# Hitung akurasi
akurasi = accuracy_score(y_test, y_pred)
print(f'Akurasi Model: {akurasi:.2%}')
print(f'Artinya: dari {len(y_test):,} prediksi, {int(akurasi*len(y_test)):,} benar')

### Confusion Matrix — Melihat Detail Kesalahan

Akurasi saja tidak cukup! Kita perlu tahu **jenis kesalahan** apa yang dibuat model.

**Confusion Matrix** adalah tabel yang menunjukkan:

```
                    Prediksi: Tidak    Prediksi: Ya
Asli: Tidak     ✅ True Negative     ⚠️ False Positive
Asli: Ya        ❌ False Negative     ✅ True Positive
```

**Analogi Deteksi COVID:**
- **True Positive** = Orang sakit, tes bilang sakit ✅ (benar!)
- **True Negative** = Orang sehat, tes bilang sehat ✅ (benar!)
- **False Positive** = Orang sehat, tes bilang sakit ⚠️ (alarm palsu)
- **False Negative** = Orang sakit, tes bilang sehat ❌ (bahaya! terlewat)

In [ ]:
# Tampilkan Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 6))
cm_display = ConfusionMatrixDisplay(cm, display_labels=['Tidak (0)', 'Ya (1)'])
cm_display.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.show()

# Penjelasan
tn, fp, fn, tp = cm.ravel()
print(f'\n📊 Rincian Prediksi:')
print(f'  ✅ Benar prediksi "Tidak": {tn:,} (True Negative)')
print(f'  ✅ Benar prediksi "Ya":    {tp:,} (True Positive)')
print(f'  ⚠️ Salah prediksi "Ya":    {fp:,} (False Positive — alarm palsu)')
print(f'  ❌ Salah prediksi "Tidak":  {fn:,} (False Negative — terlewat)')

### Precision, Recall, dan F1-Score

Tiga metrik penting untuk klasifikasi:

| Metrik | Pertanyaan yang Dijawab | Rumus Sederhana |
|--------|------------------------|------------------|
| **Precision** | Dari yang diprediksi "Ya", berapa yang benar? | TP / (TP + FP) |
| **Recall** | Dari yang aslinya "Ya", berapa yang tertangkap? | TP / (TP + FN) |
| **F1-Score** | Keseimbangan antara Precision dan Recall | Rata-rata harmonis |

**Analogi Memancing:**
- **Precision** = Dari semua ikan yang kamu tangkap, berapa yang memang ikan target? (bukan sampah)
- **Recall** = Dari semua ikan target di kolam, berapa yang berhasil kamu tangkap?

In [ ]:
# Hitung Precision, Recall, F1-Score
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print('📊 Metrik Evaluasi Detail')
print('=' * 45)
print(f'  Precision : {prec:.3f} ({prec*100:.1f}%)')
print(f'  Recall    : {rec:.3f} ({rec*100:.1f}%)')
print(f'  F1-Score  : {f1:.3f} ({f1*100:.1f}%)')
print('=' * 45)

if prec > rec:
    print('\n💡 Precision > Recall: Model berhati-hati, jarang salah prediksi "Ya",') 
    print('   tapi mungkin melewatkan beberapa nasabah yang sebenarnya mau buka deposito.')
else:
    print('\n💡 Recall > Precision: Model agresif menangkap nasabah potensial,')
    print('   tapi kadang salah mengira yang sebenarnya tidak tertarik.')

### ROC Curve — Visualisasi Performa Model

**ROC Curve** menunjukkan trade-off antara:
- **True Positive Rate** (Recall) — seberapa baik menangkap yang "Ya"
- **False Positive Rate** — seberapa sering salah alarm

**AUC (Area Under Curve)** = luas area di bawah kurva:
- **1.0** = sempurna
- **0.5** = sama saja dengan tebak acak (garis diagonal)
- **> 0.8** = model cukup bagus

In [ ]:
# Hitung ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba[:, 1])
auc = roc_auc_score(y_test, y_pred_proba[:, 1])

# Plot
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#FF6F00', linewidth=2,
         label=f'Logistic Regression (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--',
         label='Tebak Acak (AUC = 0.5)')

plt.fill_between(fpr, tpr, alpha=0.1, color='#FF6F00')

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('ROC Curve — Seberapa Bagus Model Kita?', fontsize=14, fontweight='bold')
plt.legend(fontsize=11, loc='lower right')
plt.tight_layout()
plt.show()

print(f'AUC Score: {auc:.3f}')
if auc > 0.9:
    print('✅ Excellent! Model sangat bagus.')
elif auc > 0.8:
    print('✅ Model cukup bagus.')
elif auc > 0.7:
    print('⚠️ Model lumayan, masih bisa ditingkatkan.')
else:
    print('❌ Model kurang bagus, perlu perbaikan.')

---

## 🌟 BONUS: Menangani Data Tidak Seimbang dengan SMOTE

Ingat tadi data kita **tidak seimbang** (~89% Tidak, ~11% Ya)? Ini membuat model cenderung selalu menebak "Tidak".

**SMOTE** (Synthetic Minority Over-sampling Technique) adalah teknik untuk **menyeimbangkan data** dengan membuat data sintetis (buatan) untuk kelas minoritas.

**Analogi:** Bayangkan kelas yang hanya punya 3 siswa perempuan dan 27 laki-laki. Untuk membuat diskusi lebih seimbang, kita "mengundang" teman perempuan dari kelas lain (data sintetis) sehingga jumlahnya setara.

In [ ]:
# Install imblearn jika belum ada
!pip install -q imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE

# Terapkan SMOTE pada data training (JANGAN pada data test!)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print('Sebelum SMOTE:')
print(f'  Tidak (0): {sum(y_train == 0):,}')
print(f'  Ya (1):    {sum(y_train == 1):,}')
print(f'\nSesudah SMOTE:')
print(f'  Tidak (0): {sum(y_train_smote == 0):,}')
print(f'  Ya (1):    {sum(y_train_smote == 1):,}')
print('\n✅ Data sekarang seimbang!')

In [ ]:
# Latih ulang model dengan data yang seimbang
model_smote = LogisticRegression(max_iter=1000, random_state=42)
model_smote.fit(X_train_smote, y_train_smote)

# Prediksi
y_pred_smote = model_smote.predict(X_test_scaled)
y_pred_proba_smote = model_smote.predict_proba(X_test_scaled)

# Bandingkan metrik
print('📊 Perbandingan: Tanpa SMOTE vs Dengan SMOTE')
print('=' * 55)
print(f'{"Metrik":<15} {"Tanpa SMOTE":>15} {"Dengan SMOTE":>15}')
print('-' * 55)
print(f'{"Accuracy":<15} {accuracy_score(y_test, y_pred):>15.3f} {accuracy_score(y_test, y_pred_smote):>15.3f}')
print(f'{"Precision":<15} {precision_score(y_test, y_pred):>15.3f} {precision_score(y_test, y_pred_smote):>15.3f}')
print(f'{"Recall":<15} {recall_score(y_test, y_pred):>15.3f} {recall_score(y_test, y_pred_smote):>15.3f}')
print(f'{"F1-Score":<15} {f1_score(y_test, y_pred):>15.3f} {f1_score(y_test, y_pred_smote):>15.3f}')
print(f'{"AUC":<15} {roc_auc_score(y_test, y_pred_proba[:,1]):>15.3f} {roc_auc_score(y_test, y_pred_proba_smote[:,1]):>15.3f}')
print('=' * 55)
print('\n💡 Perhatikan: SMOTE biasanya meningkatkan Recall (menangkap lebih banyak kelas "Ya")')
print('   tapi bisa menurunkan Precision (lebih banyak alarm palsu).')

---

## 📝 Ringkasan: Apa yang Kita Pelajari Hari Ini?

### Konsep Utama

1. **Logistic Regression** memprediksi **probabilitas** suatu kejadian (0%-100%), lalu memutuskan Ya/Tidak
2. Data teks harus diubah menjadi angka (**One-Hot Encoding**) sebelum dimasukkan ke model
3. **Scaling** menyamakan skala fitur agar model bekerja optimal
4. Evaluasi klasifikasi membutuhkan lebih dari sekedar akurasi:
   - **Confusion Matrix** menunjukkan jenis kesalahan
   - **Precision** = ketepatan prediksi positif
   - **Recall** = kelengkapan menangkap positif
   - **AUC-ROC** = performa keseluruhan model
5. Data tidak seimbang bisa ditangani dengan **SMOTE**

### Kapan Menggunakan Logistic Regression?

✅ **Gunakan jika:**
- Target berupa **kategori** (ya/tidak, spam/bukan spam)
- Kamu butuh model yang **mudah dipahami** dan **cepat**
- Kamu butuh **probabilitas**, bukan hanya label

❌ **Kurang cocok jika:**
- Hubungan antar fitur sangat **kompleks dan non-linear**
- Batas keputusan tidak bisa digambar sebagai garis lurus

---

### 🔜 Notebook Selanjutnya: Decision Tree
Di notebook berikutnya, kita akan belajar model yang bekerja seperti **diagram alur keputusan** — mudah divisualisasi dan dipahami!